Import Libraries

In [1]:
import numpy as np
import math

Initialize Vectors

In [2]:
L, dk, dv = 4, 8, 8
q = np.random.randn(L, dk) # querry vector
k = np.random.randn(L, dk) # key vector
v = np.random.randn(L, dv) # value vector

In [3]:
print("Q:\n", q)
print("K:\n", k)
print("V:\n", v)

Q:
 [[-0.03526891 -0.25690485  0.29242108 -2.73507618  0.07876091  1.39056469
  -0.27341646  0.61592536]
 [-0.05178431  1.47873343 -0.83520452  1.35133097  0.04676595  0.01063771
  -1.23831794 -1.28253013]
 [-1.13193019 -0.40293906 -0.05136411  0.58811229 -1.66839488  0.03481836
  -0.75599989 -0.08935215]
 [-0.20564848  1.20492177  0.87373377  1.39299465  2.31445093 -0.91616053
   0.01722847  1.05159641]]
K:
 [[-0.0135696   0.94648829  0.031781   -0.80684415  0.17814264 -0.28507242
  -0.40134255  0.0437623 ]
 [ 0.46388078  1.62028011 -0.9152096   0.65407811 -0.09931734 -0.21024473
   0.40823618 -0.52010957]
 [ 0.81434075 -0.66946985 -0.92103995 -1.71276642 -0.9499921   0.23126523
  -1.92841489  0.76223526]
 [ 1.34884182 -0.8426252  -0.15980343 -0.40404105  0.9574361  -0.75141295
   0.53731425 -1.42795409]]
V:
 [[ 0.16317879  0.9516044  -0.24248413 -0.93805007 -0.12735733  1.54074555
  -0.78230408 -0.14307789]
 [-0.23988369  0.88193144 -0.19517314  0.34471743  0.97687092 -0.36599827
  -

Self Attention

In [7]:
prod = np.matmul(q, k.T)

why do we need sqrt(dk) in the denominator

In [8]:
print("var_Q: ", np.round(q.var(), 2), ", var_K: ", np.round(k.var(), 2), ", var_Prod: ", np.round(prod.var(), 2))
# In order to reduce the variance of the Q.K_transpose product, the sqrt(dk) is included in the denominator.

var_Q:  1.08 , var_K:  0.71 , var_Prod:  7.38


In [11]:
scaled_prod = np.matmul(q, k.T) / math.sqrt(dk)
print("var_Q: ", np.round(q.var(), 2), ", var_K: ", np.round(k.var(), 2), ", var_Prod: ", np.round(prod.var(), 2), ", sc_var_Prod: ", np.round(scaled_prod.var(), 2))


var_Q:  1.08 , var_K:  0.71 , var_Prod:  7.38 , sc_var_Prod:  0.92


In [12]:
scaled_prod

array([[ 0.61083483, -1.13891809,  2.05131384, -0.2717581 ],
       [ 0.2579565 ,  1.47603331, -0.42743233, -0.18582099],
       [-0.30044896, -0.30053733,  0.48464264, -1.1733888 ],
       [ 0.26857815,  0.49187511, -2.0530813 , -0.20617747]])

Masking
- To ensure words don't get context from words generated in the future.
- Not required in encoder, but required in decoder.

In [18]:
mask = np.tril(np.ones((L, L))) # Lower triangular matrix
mask

array([[1., 0., 0., 0.],
       [1., 1., 0., 0.],
       [1., 1., 1., 0.],
       [1., 1., 1., 1.]])

In [19]:
# Arrangement for softmax function.
mask[mask == 0] = -np.Infinity
mask[mask == 1] = 0
# Because in decoder, the words must not derive content from words coming in the future.

In [20]:
scaled_prod + mask

array([[ 0.61083483,        -inf,        -inf,        -inf],
       [ 0.2579565 ,  1.47603331,        -inf,        -inf],
       [-0.30044896, -0.30053733,  0.48464264,        -inf],
       [ 0.26857815,  0.49187511, -2.0530813 , -0.20617747]])

Softmax

In [21]:
def softmax(x):
    return (np.exp(x).T / np.sum(np.exp(x), axis=1)).T


Attention

In [23]:
attention = softmax(scaled_prod + mask)
print(attention)

[[1.         0.         0.         0.        ]
 [0.22827507 0.77172493 0.         0.        ]
 [0.23852006 0.23849899 0.52298095 0.        ]
 [0.33666186 0.42089188 0.03303011 0.20941615]]


In [24]:
new_v = np.matmul(attention, v)
print(new_v)

[[ 0.16317879  0.9516044  -0.24248413 -0.93805007 -0.12735733  1.54074555
  -0.78230408 -0.14307789]
 [-0.14787457  0.89783604 -0.20597306  0.05189359  0.72480314  0.06926381
  -0.75017377 -0.52132306]
 [-0.07621684  0.00296239 -0.80678511  0.37454289 -0.04186122  0.17924104
   0.12886771  0.7657688 ]
 [ 0.18475291  0.71145344 -0.58354151 -0.13018569  0.40201461  0.24677148
  -0.53811613 -0.34093365]]


In [25]:
v

array([[ 0.16317879,  0.9516044 , -0.24248413, -0.93805007, -0.12735733,
         1.54074555, -0.78230408, -0.14307789],
       [-0.23988369,  0.88193144, -0.19517314,  0.34471743,  0.97687092,
        -0.36599827, -0.74066967, -0.63320741],
       [-0.11076167, -0.83053524, -1.3430684 ,  0.98678909, -0.46744851,
        -0.19306146,  0.94097481,  1.8182595 ],
       [ 1.11949536,  0.2259588 , -1.7925927 ,  0.03790282,  0.23481381,
        -0.5325102 ,  0.02825552, -0.412147  ]])